# BookWise — Collaborative Filtering (SVD)
Dataset: GoodBooks-10k
Metode: SVD (Singular Value Decomposition) via scikit-surprise + GridSearchCV

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from surprise import SVD, NMF, Dataset, Reader, accuracy
from surprise.model_selection import cross_validate, GridSearchCV, train_test_split
import joblib, os

DATA_DIR  = '../../data'
MODEL_DIR = '../model'

# GoodBooks-10k
ratings = pd.read_csv(f'{DATA_DIR}/ratings.csv')
ratings = ratings[ratings['rating'].between(1, 5)]

# Filter user & buku aktif
user_counts = ratings['user_id'].value_counts()
book_counts = ratings['book_id'].value_counts()
ratings = ratings[ratings['user_id'].isin(user_counts[user_counts >= 3].index)]
ratings = ratings[ratings['book_id'].isin(book_counts[book_counts >= 5].index)]

print(f'Filtered ratings: {len(ratings):,}')
print(f'Unique users: {ratings["user_id"].nunique():,}')
print(f'Unique books: {ratings["book_id"].nunique():,}')

Filtered ratings: 965,152
Unique users: 45,122
Unique books: 10,000


In [3]:
# Load ke format Surprise (rating scale 1-5)
reader = Reader(rating_scale=(1, 5))
data   = Dataset.load_from_df(ratings[['user_id', 'book_id', 'rating']], reader)

# Hyperparameter tuning dengan GridSearchCV
param_grid = {
    'n_epochs': [20, 30],
    'lr_all':   [0.005, 0.01],
    'reg_all':  [0.02, 0.1],
}
print('Running GridSearchCV...')
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs.fit(data)

print(f'Best RMSE: {gs.best_score["rmse"]:.4f}')
print(f'Best MAE:  {gs.best_score["mae"]:.4f}')
print(f'Best params: {gs.best_params["rmse"]}')

Running GridSearchCV...
Best RMSE: 0.8322
Best MAE:  0.6495
Best params: {'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}


In [ ]:
# Cross-validation dengan best params
best_params = gs.best_params['rmse']
cv_results  = cross_validate(SVD(**best_params), data, measures=['rmse', 'mae'], cv=5, verbose=True)

print(f'\nMean RMSE: {cv_results["test_rmse"].mean():.4f} ± {cv_results["test_rmse"].std():.4f}')
print(f'Mean MAE:  {cv_results["test_mae"].mean():.4f} ± {cv_results["test_mae"].std():.4f}')

# Plot RMSE per fold
plt.figure(figsize=(8, 4))
plt.plot(range(1, 6), cv_results['test_rmse'], marker='o', label='RMSE', color='steelblue')
plt.plot(range(1, 6), cv_results['test_mae'],  marker='s', label='MAE',  color='coral')
plt.xlabel('Fold')
plt.ylabel('Error')
plt.title('Cross-Validation Results (SVD) — GoodBooks-10k')
plt.legend()
plt.tight_layout()
plt.savefig('cv_results.png', dpi=150)
plt.show()

In [ ]:
# Bandingkan SVD vs NMF
print('Comparing SVD vs NMF...')
for algo_class, name in [(SVD, 'SVD'), (NMF, 'NMF')]:
    res = cross_validate(algo_class(), data, measures=['rmse', 'mae'], cv=3, verbose=False)
    print(f'{name} — RMSE: {res["test_rmse"].mean():.4f} | MAE: {res["test_mae"].mean():.4f}')

In [ ]:
# Train final model dengan best params
trainset = data.build_full_trainset()
cf_model = SVD(**best_params)
cf_model.fit(trainset)
joblib.dump(cf_model, f'{MODEL_DIR}/collaborative.pkl')
print('✅ Collaborative (SVD) model saved.')